<div align="right" style="text-align: right"><i>Peter Norvig</i></div>

# (How to Write a (Lisp) Interpreter (in Python))

This notebook describes how to implement computer language interpreters in general, and in particular  an interpreter for most of the [**Scheme**](https://www.scheme.org/) dialect of [**Lisp**](https://en.wikipedia.org/wiki/Lisp_(programming_language%29). I call my language and interpreter **Lispy** because it is Lisp implemented in Python  (lis.py). Years ago, I showed how to write a semi-practical near-complete Scheme interpreter (one in [Java](https://norvig.com/jscheme.html)  and one in [Common Lisp](https://github.com/norvig/paip-lisp/blob/main/docs/chapter22.md)). This time the goal is simplicity. 

Why should interpreters and compilers matter to you? As [Steve Yegge said](https://steve-yegge.blogspot.com/2007/06/rich-programmer-food.html?), "If you don't know how compilers work, then you don't know how computers work." Yegge describes 8 problems that can be solved with compilers (or equally well with interpreters, or with Yegge's typical heavy dosage of cynicism).

## Syntax and Semantics of  Programs

The syntax of a language is the arrangement of characters to form correct statements or expressions. For example, in the language of mathematical expressions (and in many programming languages), the syntax for adding one plus two is "1 + 2". The semantics of a language determines what it means.  We say we are evaluating an expression when we determine its value; we would say that "1 + 2" evaluates to 3, and write that as "1 + 2" ⇒ 3. 

Scheme syntax is different from most other programming languages. Consider:

     Java	      	                               |   Scheme
     ----------------------------------------------+-------------------------------
     if (x.val() > 0) {                            | (if (> (val x) 0)
         return fn(A[i] + 3 * i,                   |     (fn (+ (aref A i) (* 3 i))
                   new String[] {"one", "two"});   |         (quote (one two))))
     }                                             |

Java has a wide variety of syntactic conventions (keywords, infix operators, three kinds of brackets, operator precedence, dot notation, quotes, commas, semicolons), but Scheme syntax is much simpler:
Scheme programs consist solely of expressions; there is no statement/expression distinction.
Numbers (e.g. 1) and symbols (e.g. A) are called atomic expressions; they cannot be broken into pieces. These are similar to their Java counterparts, except that in Scheme, operators such as `+` and `>` are symbols too, and are treated the same way as `A` and `fn`.
Everything else is a list expression: a "(", followed by zero or more expressions, followed by a ")". The first element of the list determines what it means:
- A list starting with a keyword, e.g. `(if ...)`, is a **special form**; the meaning depends on the keyword.
- A list starting with a non-keyword, e.g. `(sqrt x)`, is a function call: the function `sqrt` is applied to the argument, `x`, to compute a value.

The beauty of Scheme is that the full language only needs 5 keywords and 8 syntactic forms. In comparison, Python has 33 keywords and 110 syntactic forms, and Java has 50 keywords and 133 syntactic forms. All those parentheses may seem intimidating, but Scheme syntax has the virtues of simplicity and consistency. (Some have joked that "Lisp" stands for "**L**ots of **I**rritating **S**illy **P**arentheses"; I think it stand for "**L**isp **I**s **S**yntactically **P**ure".)


How do we go from syntax to semantics? Here is the traditional approach:
- **Parsing**: A function called `parse` takes the program (a sequence of characters) as input and transforms it into an internal representation, called an **abstract syntax tree**, that closely mirrors the nested structure of statements or expressions in the program. This will be done in two steps. First,  the function`tokenize` breaks up the characters into tokens (symbols such as parentheses, numbers such as `123`, and symbols such as `if`).Second, the function `read_from_tokens` converts the tokens into the abstract syntax tree.
- **Execution**: A function called `eval` evaluates the syntax tree to produce the value of the program. 

# Language 1: Lispy Calculator

We won't tackle all of Scheme right away; instead we'll start with a subset of Scheme I call **Lispy Calculator**. Lispy Calculator lets you do any computation you could do on a typical calculator—as long as you are comfortable with prefix notation. And you can do two things that are not offered in most calculators: use an `if` to define a conditional expression, and use a `define` to set the value of a new variable. Here's an example program, that computes the area of a circle of radius 10, using the formula π r<sup>2</sup>

     (begin
       (define r 10)
       (* pi (* r r)))


Here is a table of all the allowable expressions in the Lispy Calculator Language. In the Syntax column of this table, *symbol* must be a symbol, *number* must be an integer or floating point number, and the other italicized words can be any expression. The notation *arg...* means zero or more repetitions of *arg*.

|Expression	|Syntax	|Example|Semantics|
|-----------|-------|-------|-------------|
|constant literal	|*number*	|`12` or `-3.45e+6`|A number evaluates to itself.|
|variable reference|	*symbol*	|`r`|A symbol is interpreted as a variable name; its value is the variable's value.|
|conditional	|(`if` *test then_part else_part*`)`|`(if (< x 0) (- x) x)`|	Evaluate test; if true, evaluate and return conseq; otherwise alt.|
|definition	|`(define` *symbol exp*`}`	|`(define r 10)`|Define a new variable and give it the value of evaluating the expression exp.|
|procedure call	|`(`*proc arg*...`)`	|`(sqrt (* 2 8))` ⇒ 4.0|Evaluate proc and all the args, and then the procedure is applied to the list of arg values.|


Let's get some imports out of the way, and be explicit about our representations for Scheme object types and their representation in Python:

In [1]:
import numbers
import math
import operator as op

Number = numbers.Number    # A Scheme Number is implemented as a Python number (e.g., int or float)
Symbol = str               # A Scheme Symbol is implemented as a Python str
List   = list              # A Scheme List is implemented as a Python list
Atom   = Symbol | Number   # A Scheme Atom is a Symbol or Number
Exp    = Atom | List       # A Scheme expression is an Atom or List
Env    = dict[Symbol, Exp] # A Scheme environment is a dictionary mapping of {variable: value}

## Evaluation: eval 

Here is the core of the interpreter, `eval`. It takes as input an expression, `exp`, and an **environment** that specifies the values of variables. It returns the value of the expression, given the environment. These few lines are what  what [Alan Kay called](https://queue.acm.org/detail.cfm?id=1039523) "the Maxwell's Equations of Software."


In [2]:
def eval(exp: Exp, env: Env) -> object:
    """Evaluate an expression in an environment."""
    match exp:
        case Number():                         # number evaluates to itself 
            return exp
        case Symbol():                         # variable evaluates to its value in environment
            return env[exp]
        case ('if', test, then, els):          # conditional evaluates one branch or the other
            branch = (then if eval(test, env) 
                      else els)
            return eval(branch, env)
        case ('define', name, exp):            # definition adds name to the environment
            env[name] = eval(exp, env)
        case (proc, *args):                    # procedure call
            func = eval(proc, env)
            vals = [eval(arg, env) for arg in args]
            return func(*vals)

## Parsing: parse, tokenize and read_from_tokens

How do we get from a sequence of characters to the abstract syntax tree that `eval` expects? The function `parse` does the job, in two steps: 
1. **Lexical analysis**: the function `tokenize` breaks the characters into tokens (such as the keyword `"begin"` or the number `"10"`).
2. **Syntactic analysis**: the function `read_from_tokens` converts the tokens into an expression.

There are many tools for lexical analysis (such as Mike Lesk and Eric Schmidt's [lex](https://en.wikipedia.org/wiki/Lex_(software%29)), but we will use a very simple tool: Python's `str.split`: 

In [3]:
def tokenize(chars: str) -> list:
    """Convert a string of characters into a list of tokens.
    (Put spaces around parens, then split on spaces.)"""
    return chars.replace('(', ' ( ').replace(')', ' ) ').split()

`read_from_tokens` looks at the first token; if it is a `)` that's a syntax error. If it is a `(`, then we start building up a list of sub-expressions until we hit a matching ')'. Any non-parenthesis token must be an atom. First try to interpret it as a number, and failing that, it must be a symbol. 

In [4]:
def read_from_tokens(tokens: list[str]) -> Exp:
    """Read an expression from a list of tokens, mutating the list."""
    token = tokens.pop(0) if tokens else None
    match token:
        case None:
            raise SyntaxError('unexpected end of expression')
        case ')':
            raise SyntaxError('unexpected ")"')
        case '(':
            result = []
            while tokens[0] != ')':
                result.append(read_from_tokens(tokens))
            tokens.pop(0) # pop off ')'
            return result
        case _:
            try:
                n = float(token)
                return int(n) if n.is_integer() else n
            except ValueError:
                return token # symbol

Now we're ready to parse a sample program:

In [5]:
def parse(program: str) -> Exp:
    """Read a Scheme expression from a string.
    (First split the program string into tokens, then read from the token list."""
    return read_from_tokens(tokenize(program))

program = "(begin (define r 10) (* pi (* r r)))"
parse(program)

['begin', ['define', 'r', 10], ['*', 'pi', ['*', 'r', 'r']]]

We can also see the tokenization of the program:

In [6]:
tokenize(program)

['(',
 'begin',
 '(',
 'define',
 'r',
 '10',
 ')',
 '(',
 '*',
 'pi',
 '(',
 '*',
 'r',
 'r',
 ')',
 ')',
 ')']

## Environments

We mentioned in passing that an **environment** is a mapping from variable names to their values. We will define a default global environment with the names for a bunch of standard functions (like `sqrt` and `max`, and also operators like `+` and `>`). This environment can be augmented with user-defined variables, using the expression `(define symbol value)`.

In [7]:
global_env = Env({
    '+':op.add, '-':op.sub, '*':op.mul, '/':op.truediv, 
    '>':op.gt,  '<':op.lt,  '>=':op.ge, '<=':op.le, '=':op.eq, 
    'abs':     abs,
    'append':  op.add,  
    'apply':   lambda proc, args: proc(*args),
    'begin':   lambda *x: x[-1],
    'car':     lambda x: x[0],
    'cdr':     lambda x: x[1:], 
    'cons':    lambda x,y: [x] + y,
    'eq?':     op.is_, 
    'equal?':  op.eq,
    'expt':    pow,
    'length':  len, 
    'list':    lambda *x: List(x), 
    'list?':   lambda x: isinstance(x, list), 
    'map':     map,
    'max':     max, 'min': min,
    'not':     op.not_,
    'null?':   lambda x: x == [], 
    'number?': lambda x: isinstance(x, Number),  
	'print':   print,
    'procedure?': callable,
    'round':   round,
    'symbol?': lambda x: isinstance(x, Symbol),
    **vars(math)})

We're done! You can see it all in action:

In [8]:
eval(parse(program), global_env)

314.1592653589793

## Interaction: A REPL

It is tedious to have to enter `eval(parse("...", global_env))` all the time. One of Lisp's great legacies is the notion of an interactive read-eval-print loop: a way for a programmer to enter an expression, and see it immediately read, evaluated, and printed, without having to go through a lengthy build/compile/run cycle. So let's define the function `repl` (which stands for read-eval-print-loop), and the function `scheme_to_str` which returns a string representing a Scheme object.

In [9]:
def repl(prompt='\nlis.py> '):
    """A prompt-read-eval-print loop."""
    print('Scheme read-eval-print loop. Type exit to exit')
    while (expr := parse(input(prompt))) != 'exit':
        val = eval(expr, global_env)
        if val is not None: 
            print(scheme_to_str(val))

def scheme_to_str(exp):
    "Convert a Python object back into a Scheme-readable string."
    if isinstance(exp, List):
        return '(' + ' '.join(map(scheme_to_str, exp)) + ')' 
    else:
        return str(exp)

Here is `repl` in action. You can run it yourself in a cell.

    >>> repl()
    lis.py> (define r 10)
    lis.py> (* pi (* r r))
    314.159265359
    lis.py> (if (> (* 11 11) 120) (* 7 6) oops)
    42
    lis.py> (list (+ 1 1) (+ 2 2) (* 2 3) (expt 2 3))
    lis.py> (2 4 6 8)

# Language 2: Full Lispy

We will now extend our language with three new special forms, giving us a much more nearly-complete Scheme subset:

|Expression	|Syntax| Example|	Semantics|
|-----------|------|--------|------------|
|quotation	|`(quote` *exp*`)`| `(quote (+ 1 2))` ⇒ `(+ 1 2)`|	Return the exp literally; do not evaluate it.|
|assignment	|`(set!` *symbol exp*`)`| `(set! r2 (* r r))`|	Evaluate *exp* and assign that value to *symbol*.|
|procedure	|`(lambda (`*symbol...*`)` *exp*`)`|`(lambda (r) (* pi (* r r)))`|	Create a procedure with parameter(s) named *symbol...* and *exp* as the body.|

The lambda special form (an obscure nomenclature choice that refers to Alonzo Church's [lambda calculus](https://en.wikipedia.org/wiki/Lambda_calculus)) creates a procedure. We want procedures to work like this:

    lis.py> (define circle-area (lambda (r) (* pi (* r r)))
    lis.py> (circle-area (+ 5 5))
    314.159265359


There are two steps here. In the first step, the lambda expression is evaluated to create a procedure, one which refers to the global variables pi and `*`, and takes a single parameter, which it calls `r`. This procedure is defined as the value of the new variable `circle-area`. In the second step, the procedure we just defined is the value of circle-area, so it is called, with the value 10 as the argument. We want `r` to take on the value 10, but it wouldn't do to just set `r` to 10 in the global environment. What if we were using `r` for some other purpose? We wouldn't want a call to `circle-area` to alter that value. Instead, we want to arrange for there to be a **local variable** named `r `that we can set to 10 without worrying about interfering with any global variable that happens to have the same name. The process for calling a procedure introduces these new local variable(s), binding each symbol in the parameter list of the function to the corresponding value in the argument list of the function call.

The difference between `set!` and `define` is that `set!` always refers to a previously-defined variable, whcih might be in the innermost environment, or might be in an outer environment. `define` always introduces a new variable in the innermost environment.


## Redefining Env as a Class

To handle local variables, we will redefine `Env` to be a subclass of `dict.` When we evaluate (`circle-area (+ 5 5))`, we will fetch the procedure body, `(* pi (* r r))`, and evaluate it in an environment that has `r` as the sole local variable (with value 10), but also has the global environment as the "outer" environment; it is there that we will find the values of `*` and `pi`.  In the diagram, the inner environment is blue and the outer red:

<p><table  style="border: 3px solid red" cellspacing=1 cellpadding=5><tr><td>
<tt>pi: 3.141592653589793
<br>*: &lt;built-in function mul&gt;
<br>...
<br>
<table  style="border: 3px solid blue" cellspacing=1 cellpadding=5>
<tr><td>r: 10
</table>
</table>

When we look up a variable in such a nested environment, we look first at the innermost level, but if we don't find the variable name there, we move to the next outer level. Procedures and environments are intertwined, so let's define them together:

In [10]:
from dataclasses import dataclass

class Env(dict):
    """An environment: a dict of {'var': val} pairs, with an outer Env."""
    def __init__(self, inner, outer=None):
        self.update(inner)
        self.outer = outer
    def find(self, var):
        """Find the innermost Env where var appears."""
        return self if (var in self or not self.outer) else self.outer.find(var)

@dataclass
class Procedure(object):
    """A user-defined Scheme procedure."""
    parms: list[Symbol]
    body:  Exp
    env:   Env
    def __call__(self, *args): 
        return eval(self.body, Env(zip(self.parms, args), outer=self.env))

# Now transfer the old global_env into a new Env
global_env = Env(global_env)

We see that every procedure has three components: a list of parameter names, a body expression, and an environment that tells us what other variables are accessible from the body. For a procedure defined at the top level this will be the global environment, but it is also possible for a procedure to refer to the local variables of the environment in which it was defined (**not** the environment in which it is called).

An environment is a subclass of `dict`, so it has all the methods that `dict` has. In addition there are two methods: the constructor `__init__` builds a new environment by taking a list of parameter names and a corresponding list of argument values, and creating a new environment that has those {variable: value} pairs as the inner part, and also refers to the given outer environment. The method `find` is used to find the right environment for a variable: starting with the inner one and going out, find the first environment that mentions the variable name.

To see how these all go together, here is the new definition of `eval`. Note that the clause for variable reference has changed: we now have to call `env.find(exp)` to find at what level the variable exists; then we can fetch the value of the variable from that level. (The clause for `define` has not changed, because a define always adds a new variable to the innermost environment.) There are three new clauses: `quote` returns the constant without evaluating it,  `set!`, finds the environment level where the variable exists and sets it to a new value there,  and for `lambda` we create a new procedure object with the given parameter list, body, and environment.

In [11]:
def eval(exp: Exp, env=global_env) -> object:
    """Evaluate an expression in an environment."""
    match exp:
        case Symbol():                             # variable reference
            return env.find(exp)[exp]
        case Number():                             # constant 
            return exp
        case ('if', test, then, els):              # conditional evaluates one branch or the other
            branch = (then if eval(test, env) 
                      else els)
            return eval(branch, env)
        case ('define', name, exp):                # definition
            env[name] = eval(exp, env)
        case ('quote', constant):
            return constant
        case ('set!', symbol, exp):
            env.find(symbol)[symbol] = eval(exp, env)
        case ('lambda', parms, body):
            return Procedure(parms, body, env)
        case (proc, *args):                        # procedure call
            func = eval(proc, env)
            vals = [eval(arg, env) for arg in args]
            return func(*vals)

You can run try new version by removing the `#` and running the cell below:

In [12]:
# repl()

Here is an example of what you can do:

    >>> repl()
    
    lis.py> (define circle-area (lambda (r) (* pi (* r r))))
    
    lis.py> (circle-area 3)
    28.274333877
    
    lis.py> (define fact (lambda (n) (if (<= n 1) 1 (* n (fact (- n 1))))))
    
    lis.py> (fact 10)
    3628800
    
    lis.py> (fact 100)
    9332621544394415268169923885626670049071596826438162146859296389521759999322991
    5608941463976156518286253697920827223758251185210916864000000000000000000000000
    
    lis.py> (circle-area (fact 10))
    4.1369087198e+13
    
    lis.py> (define first car)
    
    lis.py> (define rest cdr)
    
    lis.py> (define count (lambda (item L) (if L (+ (equal? item (first L)) (count item (rest L))) 0)))
    
    lis.py> (count 0 (list 0 1 2 3 0 0))
    3
    
    lis.py> (count (quote the) (quote (the more the merrier the bigger the better)))
    4
    
    lis.py> (define twice (lambda (x) (* 2 x)))
    
    lis.py> (twice 5)
    10
    
    lis.py> (define repeat (lambda (f) (lambda (x) (f (f x)))))
    
    lis.py> ((repeat twice) 10)
    40
    
    lis.py> ((repeat (repeat twice)) 10)
    160
    
    lis.py> ((repeat (repeat (repeat twice))) 10)
    2560
    
    lis.py> ((repeat (repeat (repeat (repeat twice)))) 10)
    655360
    
    lis.py> (pow 2 16)
    65536.0
    
    lis.py> (define fib (lambda (n) (if (< n 2) 1 (+ (fib (- n 1)) (fib (- n 2))))))
    
    lis.py> (define range (lambda (a b) (if (= a b) (quote ()) (cons a (range (+ a 1) b)))))
    
    lis.py> (range 0 10)
    (0 1 2 3 4 5 6 7 8 9)
    
    lis.py> (map fib (range 0 10))
    (1 1 2 3 5 8 13 21 34 55)
    
    lis.py> (map fib (range 0 20))
    (1 1 2 3 5 8 13 21 34 55 89 144 233 377 610 987 1597 2584 4181 6765)
    
    
We now have a language with procedures, variables, conditionals (`if`), and sequential execution (`begin`). If you are familiar with other languages, you might think that a while or for loop would be needed, but Scheme manages to do without these just fine. The Scheme report says "Scheme demonstrates that a very small number of rules for forming expressions, with no restrictions on how they are composed, suffice to form a practical and efficient programming language." In Scheme you iterate by defining recursive functions.

## How Small/Complete/Good/Fast is Lispy?
    
In which we judge Lispy on several criteria:
- **Small**: Lispy is *very* small: about 120 lines or 4K of source code. (An earlier version was just 90 lines, but was perhaps a bit too terse.)   The smallest version of my Scheme in Java, [Jscheme](http://norvig.com/jscheme.html) was 1664 lines and 57K of source. Jscheme was
  originally called SILK (Scheme in Fifty Kilobytes), but I only kept
  under that limit by counting bytecode rather than source code. Lispy does much
  better; I think it meets Alan Kay's 1972 [claim](http://gagne.homedns.org/~tgagne/contrib/EarlyHistoryST.html)
  that *you could define the "most powerful language in the world" in "a page of code."* (if you use a small font). However,
  I think Alan would disagree, because he would count the Python compiler as part of the code, putting me <i>well</i> over a page.
- **Complete**: Lispy is not very complete compared to the Scheme standard.  Some major shortcomings:
  - **Syntax**: Missing comments, quote and quasiquote notation, # literals, the derived
  expression types (such as `cond`, derived from `if`,
  or `let`, derived from `lambda`), and dotted list notation.
  - **Semantics**: Missing `call/cc` and tail recursion.  
  - **Data Types**: Missing strings, characters, booleans, ports,
  vectors, exact/inexact numbers. A Scheme list should actually be a custom data class, not a Python list.
  - **Procedures**: Missing over 100 primitive procedures.
  - **Error recovery**: Lispy does not attempt to detect,
  reasonably report, or recover from errors.  Lispy expects the
  programmer to be perfect. (That is an environment issue, not a language issue.)
- **Good**: That's up to the readers to decide.  I think that Lispy is good for my purpose of explaining Lisp interpreters.
- **Fast**: Lispy computes <tt>(fact 100)</tt> in about a millisecond. That's fast enough for me (although slower than some
other ways of computing it). <p>

In [13]:
fact = """(begin (define fact (lambda (n) 
                                (if (<= n 1) 
                                    1 
                                    (* n (fact (- n 1)))))) 
                 (fact 100))"""

%time eval(parse(fact))

CPU times: user 371 μs, sys: 108 μs, total: 479 μs
Wall time: 484 μs


93326215443944152681699238856266700490715968264381621468592963895217599993229915608941463976156518286253697920827223758251185210916864000000000000000000000000

## True Story

To back up the idea that it can be very helpful to know how
interpreters work, here's a story.  Way back in 1984 I was writing a
Ph.D. thesis.  This was before LaTeX, before Microsoft Word for Windows–we used
[troff](https://en.wikipedia.org/wiki/Troff). Unfortunately, troff had no facility for forward references
to symbolic labels: I wanted to be able to write "As we will see on
page @theorem-x" and then write something like "@(set theorem-x \n%)" in
the appropriate place (the troff register \n% holds the page number). My
fellow grad student Tony DeRose felt the same need, and together we
sketched out a simple Lisp program that would handle this as a preprocessor.  However,
it turned out that the Lisp we had at the time was good at reading
Lisp expressions, but slow at reading 100 KB of characters one character at a time.

From there Tony and I split paths.  He reasoned that the hard part was
the interpreter for expressions; he needed Lisp for that, but he knew
how to write a tiny C routine
for reading the characters one at a time, and how to link the C routine into the Lisp
program.  I didn't know how to do that linking, but I reasoned that writing an
interpreter for this trivial language (all it had was set variable,
fetch variable, and string concatenate) was easy, so I wrote an
interpreter in C. So, ironically, Tony wrote a Lisp program (with one small routine in C) because he was a
C programmer, and I wrote a C program (that implements a hand-coded mini-interpreter) because I was a Lisp programmer.

In the end, we both got our theses done (<a href="http://www.eecs.berkeley.edu/Pubs/TechRpts/1985/6081.html">Tony</a>, <a href="http://www.eecs.berkeley.edu/Pubs/TechRpts/1987/5995.html">Peter</a>).

<h2>Further Reading</h2>

   
  
  <p>To learn more about Scheme consult some of the fine books (by
   <a
   href="http://books.google.com/books?id=xyO-KLexVnMC&lpg=PP1&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Friedman
   and Fellesein</a>,
   <a href="http://books.google.com/books?id=wftS4tj4XFMC&lpg=PA300&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Dybvig</a>,
   <a
   href="http://books.google.com/books?id=81mFK8pqh5EC&lpg=PP1&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Queinnec</a>,
   <a href="http://www.eecs.berkeley.edu/~bh/ss-toc2.html">Harvey and
   Wright</a> or
   <a href="https://www.researchgate.net/profile/Gerald-Sussman-2/publication/37597721_Structure_and_Interpretation_of_Computer_Programs_H_Abelson_GJ_Sussman_colaboracion_de_J_Sussman_prol_de_Alan_J_Perlis/links/53d141450cf220632f392bf7/Structure-and-Interpretation-of-Computer-Programs-H-Abelson-GJ-Sussman-colaboracion-de-J-Sussman-prol-de-Alan-J-Perlis.pdf">Sussman and Abelson</a>),
   videos (by <a
   href="http://groups.csail.mit.edu/mac/classes/6.001/abelson-sussman-lectures/">Abelson
   and Sussman</a>),
   tutorials (by
      <a
   href="http://www.ccs.neu.edu/home/dorai/t-y-scheme/t-y-scheme.html">Dorai</a>,
   <a href="http://docs.racket-lang.org/quick/index.html">PLT</a>, or
   <a href="http://cs.gettysburg.edu/~tneller/cs341/scheme-intro/index.html">Neller</a>),
   or the
      <a
   href="http://www.schemers.org/Documents/Standards/R5RS/HTML">reference
   manual</a>.

<p>I also have another page describing a <a href="http://norvig.com/lispy2.html">more advanced version of Lispy</a>.